## Olist 巴西电商经营分析报告
## 物流履约优化与策略建议

---

**分析师**：Cristina Yan
**数据集**：[Olist Brazilian E-Commerce Dataset (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
**分析周期**：2017-1 ~ 2018-08
**分析工具**：Python + DuckDB + SQL + Matplotlib
**报告目的**：基于 Olist 巴西电商平台真实交易数据，深入分析平台经营现状、用户行为、物流履约效率，并提出可落地的策略优化建议

---

## 报告结构

| 章节 | 分析主题 | 核心问题 |
|:---:|---|---|
| 一 | 平台核心 KPI 概览 | 业务规模和健康度如何？ |
| 二 | 月度 GMV 与订单趋势 | 增长拐点在哪里？ |
| 三 | 订单状态漏斗 | 各状态分布与流转情况？ |
| 四 | 商品品类销售分析 | 哪些品类贡献最多？ |
| 五 | 地理分布分析 | GMV 地域集中度如何？ |
| 六 | 物流履约深度分析 | 配送时效与运费合理性？ |
| 七 | 评价与满意度分析 | 配送对评分影响几何？ |
| 八 | 留存群组分析 | 新客复购率为何偏低？ |
| 九 | RFM 用户分层 | 如何识别高价值客户？ |
| 十 | 策略建议与结论 | 优化方向与落地优先级？ |



## 数据集概览

Olist 平台数据包含 8 张核心表，覆盖从下单→支付→发货→配送→评价的完整链路：

| 表名 | 说明 | 关键字段 |
|---|---|---|
| `olist_orders` | 订单主表 | order_id, customer_id, order_status, 时间戳 |
| `olist_order_items` | 订单明细 | order_id, product_id, seller_id, price, freight_value |
| `olist_payments` | 支付信息 | order_id, payment_type, payment_value, payment_installments |
| `olist_reviews` | 买家评价 | order_id, review_score, review_comment_message |
| `olist_products` | 商品信息 | product_id, product_category_name, product_weight_g |
| `olist_customers` | 买家信息 | customer_id, customer_unique_id, customer_state |
| `olist_sellers` | 卖家信息 | seller_id, seller_state |
| `olist_category` | 品类翻译 | product_category_name, product_category_name_english |

> **口径说明**：
> - "未履约"订单 = 状态非 `delivered`
> - "有效订单" = 未取消 + 截止日（2018-08-26）前15天已签收/出库/取消
> - GMV = 支付金额（payment_value）之和，非商品标价



In [ ]:
# ============================================================
DATA_PATH = '/Users/你的路径'

# 导入分析库
import pandas as pd
import numpy as np
import duckdb
import os
import warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib

warnings.filterwarnings('ignore')

# 中文字体配置（macOS）
try:
    plt.rcParams['font.family'] = ['PingFang SC', 'Hiragino Sans GB', 'Arial Unicode MS', 'sans-serif']
except Exception:
    plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = '#FAFAFA'
plt.rcParams['axes.facecolor'] = '#FFFFFF'
plt.rcParams['figure.dpi'] = 100

# 配色方案
C_DARK_BLUE = '#1F4E79'
C_MED_BLUE  = '#2E86AB'
C_ORANGE    = '#E07B3C'
C_GREEN     = '#28A745'
C_RED       = '#DC3545'
C_GRAY      = '#6C757D'
C_GOLD      = '#F5A623'

print(f"数据路径: {DATA_PATH}")
print(f"DuckDB 版本: {duckdb.__version__}")
print(f"Matplotlib 版本: {matplotlib.__version__}")



In [ ]:
# ============================================================
# 加载数据到 DuckDB
# ============================================================
con = duckdb.connect(':memory:')

csv_files = {
    'olist_orders':      'olist_orders_dataset.csv',
    'olist_order_items': 'olist_order_items_dataset.csv',
    'olist_payments':    'olist_order_payments_dataset.csv',
    'olist_reviews':     'olist_order_reviews_dataset.csv',
    'olist_products':    'olist_products_dataset.csv',
    'olist_customers':   'olist_customers_dataset.csv',
    'olist_sellers':     'olist_sellers_dataset.csv',
    'olist_category':    'product_category_name_translation.csv',
}

print("正在加载数据表...")
for table_name, filename in csv_files.items():
    filepath = os.path.join(DATA_PATH, filename)
    con.execute(f"""
        CREATE OR REPLACE VIEW {table_name} AS
        SELECT * FROM read_csv_auto('{filepath}', header=True)
    """)
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  ✅ {table_name}: {count:,} 行")

print("\n所有数据表加载完毕！")



---
## 第一部分：平台核心 KPI 概览

**分析问题**：平台的整体业务规模和健康状况如何？

**业务背景**：Olist 成立于 2015 年，是巴西以 SaaS 模式运营的电商服务平台，核心定位是聚合中小商家接入 MercadoLivre、Americanas 等主流电商渠道。平台本身不持有库存、不负责配送，而是通过软件服务、渠道对接和声誉共享机制为商家赋能。这一商业模式意味着：物流责任归属商家，Olist 通过激励/约束机制和流量分配算法间接影响配送效率。



In [ ]:
# ============================================================
# 第一部分：平台核心 KPI 分析
# ============================================================

# --- SQL 查询：核心 KPI ---
query_kpi = """
WITH order_stats AS (
    SELECT
        COUNT(DISTINCT o.order_id)                                              AS total_orders,
        COUNT(DISTINCT o.customer_id)                                           AS unique_customers,
        ROUND(SUM(p.payment_value), 2)                                         AS total_gmv,
        ROUND(AVG(p.payment_value), 2)                                         AS avg_order_value,
        MIN(o.order_purchase_timestamp::DATE)                                  AS first_order_date,
        MAX(o.order_purchase_timestamp::DATE)                                  AS last_order_date,
        COUNT(DISTINCT o.order_id) FILTER (WHERE o.order_status = 'delivered') AS delivered_orders,
        COUNT(DISTINCT o.order_id) FILTER (WHERE o.order_status = 'canceled')  AS canceled_orders
    FROM olist_orders o
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status != 'canceled'
)
SELECT
    total_orders,
    unique_customers,
    ROUND(total_gmv / 1e6, 2)                          AS gmv_million_brl,
    avg_order_value,
    ROUND(100.0 * delivered_orders / total_orders, 1)  AS delivery_rate_pct,
    ROUND(100.0 * canceled_orders / total_orders, 1)   AS cancel_rate_pct,
    first_order_date,
    last_order_date,
    DATEDIFF('day', first_order_date, last_order_date) AS data_span_days
FROM order_stats
"""

kpi = con.execute(query_kpi).df()
print("=" * 60)
print("平台核心 KPI 概览")
print("=" * 60)
for col, val in kpi.iloc[0].items():
    print(f"  {col}: {val}")

# --- SQL：月度GMV趋势（用于验证） ---
query_gmv_check = """
SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp::DATE) AS order_month,
    ROUND(SUM(p.payment_value) / 1e6, 3) AS gmv_m,
    COUNT(DISTINCT o.order_id) AS order_count
FROM olist_orders o
JOIN olist_payments p ON o.order_id = p.order_id
WHERE o.order_status NOT IN ('canceled')
  AND o.order_purchase_timestamp IS NOT NULL
GROUP BY 1
ORDER BY 1
"""
gmv_trend = con.execute(query_gmv_check).df()

# --- 可视化：核心KPI卡片 + 月度趋势 ---
fig = plt.figure(figsize=(16, 10))

# KPI 指标卡片
ax1 = fig.add_subplot(2, 2, 1)
ax1.axis('off')
kpi_vals = [
    ("总订单量", f"{kpi.iloc[0]['total_orders']:,.0f}", "笔"),
    ("独立客户", f"{kpi.iloc[0]['unique_customers']:,.0f}", "人"),
    ("总 GMV", f"{kpi.iloc[0]['gmv_million_brl']}", "百万 BRL"),
    ("平均客单价", f"{kpi.iloc[0]['avg_order_value']:.1f}", "BRL"),
    ("完成履约率", f"{kpi.iloc[0]['delivery_rate_pct']:.1f}", "%"),
    ("取消率", f"{kpi.iloc[0]['cancel_rate_pct']:.1f}", "%"),
]
y_positions = [0.85, 0.68, 0.51, 0.34, 0.17, 0.0]
colors_card = [C_DARK_BLUE, C_MED_BLUE, C_ORANGE, C_GREEN, C_DARK_BLUE, C_RED]
for idx, ((label, value, unit), y, col) in enumerate(zip(kpi_vals, y_positions, colors_card)):
    ax1.text(0.05, y, label, fontsize=12, color='#555', va='center')
    ax1.text(0.05, y - 0.06, f"{value} {unit}", fontsize=20, fontweight='bold', color=col, va='center')

# 月度GMV趋势
ax2 = fig.add_subplot(2, 2, 2)
ax2.bar(gmv_trend['order_month'].astype(str), gmv_trend['gmv_m'], color=C_MED_BLUE, alpha=0.85, label='月度GMV')
ax2.set_title('月度 GMV 趋势（百万 BRL）', fontsize=13, fontweight='bold', pad=10)
ax2.set_xlabel('月份', fontsize=10)
ax2.set_ylabel('GMV（百万 BRL）', fontsize=10)
ax2.tick_params(axis='x', rotation=45, labelsize=8)
peak_idx = gmv_trend['gmv_m'].idxmax()
peak_month = str(gmv_trend.loc[peak_idx, 'order_month'])[:7]
ax2.bar(peak_idx, gmv_trend.loc[peak_idx, 'gmv_m'], color=C_ORANGE, label=f'峰值 {peak_month}')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# 月度订单量趋势
ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(range(len(gmv_trend)), gmv_trend['order_count'], 'o-', color=C_GREEN, linewidth=2, markersize=4)
ax3.fill_between(range(len(gmv_trend)), gmv_trend['order_count'], alpha=0.2, color=C_GREEN)
ax3.set_title('月度订单量趋势', fontsize=13, fontweight='bold', pad=10)
ax3.set_xlabel('月份序号', fontsize=10)
ax3.set_ylabel('订单量', fontsize=10)
xlabels = [str(m)[:7] for m in gmv_trend['order_month']]
ax3.set_xticks(range(0, len(gmv_trend), max(1, len(gmv_trend)//6)))
ax3.set_xticklabels([xlabels[i] for i in range(0, len(gmv_trend), max(1, len(gmv_trend)//6))], rotation=45, fontsize=8)
ax3.grid(alpha=0.3)

# 客单价月度趋势
ax4 = fig.add_subplot(2, 2, 4)
aov_by_month = gmv_trend['gmv_m'] / (gmv_trend['order_count'] / 1e6) * 1000
ax4.plot(range(len(gmv_trend)), aov_by_month, 's-', color=C_ORANGE, linewidth=2, markersize=4)
ax4.axhline(y=aov_by_month.mean(), color=C_RED, linestyle='--', label=f'均值 {aov_by_month.mean():.0f} BRL')
ax4.set_title('月度客单价趋势（BRL）', fontsize=13, fontweight='bold', pad=10)
ax4.set_xlabel('月份序号', fontsize=10)
ax4.set_ylabel('AOV（BRL）', fontsize=10)
ax4.set_xticks(range(0, len(gmv_trend), max(1, len(gmv_trend)//6)))
ax4.set_xticklabels([xlabels[i] for i in range(0, len(gmv_trend), max(1, len(gmv_trend)//6))], rotation=45, fontsize=8)
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.suptitle('Olist 平台核心经营指标', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/01_kpi_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 图表已保存：01_kpi_overview.png")



---
## 第二部分：月度 GMV 与订单趋势

**分析问题**：平台 GMV 和订单量如何随时间变化？增长拐点在哪里？

**分析发现**：
- 2017年11月（黑五）是全年分水岭：GMV 同比+137.6%，是自然月的近2倍
- AOV（客单价）稳定在 ~161 BRL，**增长完全依赖订单量扩张而非客单价提升**
- 2018年继续保持高速增长，8月达到数据期内最高峰
- 注意：数据截止至2018-08-26，缺少9-12月数据，故年末占比被低估



In [ ]:
# ============================================================
# 第二部分：月度 GMV 与订单趋势
# ============================================================

# --- SQL：月度趋势详细分析 ---
query_monthly = """
WITH monthly_data AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp::DATE) AS order_month,
        COUNT(DISTINCT o.order_id)                            AS order_count,
        ROUND(SUM(p.payment_value), 2)                       AS gmv,
        ROUND(AVG(p.payment_value), 2)                        AS aov,
        COUNT(DISTINCT o.customer_unique_id)                 AS unique_buyers
    FROM olist_orders o
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled')
      AND o.order_purchase_timestamp IS NOT NULL
    GROUP BY 1
)
SELECT
    order_month,
    order_count,
    ROUND(gmv / 1e6, 3) AS gmv_million,
    aov,
    unique_buyers,
    ROUND(order_count - LAG(order_count) OVER (ORDER BY order_month), 0) AS order_delta,
    ROUND(100.0 * (order_count - LAG(order_count) OVER (ORDER BY order_month))
          / NULLIF(LAG(order_count) OVER (ORDER BY order_month), 0), 1) AS mom_growth_pct
FROM monthly_data
ORDER BY order_month
"""

monthly = con.execute(query_monthly).df()

# 同比分析（2017 vs 2018）
query_yoy = """
WITH monthly_gmv AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp::DATE) AS month,
        SUM(p.payment_value) AS gmv
    FROM olist_orders o
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled')
      AND EXTRACT(YEAR FROM o.order_purchase_timestamp::DATE) IN (2017, 2018)
      AND o.order_purchase_timestamp IS NOT NULL
    GROUP BY 1
)
SELECT
    EXTRACT(MONTH FROM month) AS month_num,
    ROUND(SUM(CASE WHEN EXTRACT(YEAR FROM month) = 2017 THEN gmv ELSE 0 END) / 1e6, 2) AS gmv_2017,
    ROUND(SUM(CASE WHEN EXTRACT(YEAR FROM month) = 2018 THEN gmv ELSE 0 END) / 1e6, 2) AS gmv_2018
FROM monthly_gmv
GROUP BY 1
ORDER BY 1
"""

yoy = con.execute(query_yoy).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 图1：月度GMV柱状图 + 峰值标注
ax1 = axes[0, 0]
months_str = [str(m)[:7] for m in monthly['order_month']]
bars = ax1.bar(months_str, monthly['gmv_million'], color=C_MED_BLUE, alpha=0.8)
peak_idx = monthly['gmv_million'].idxmax()
bars[peak_idx].set_color(C_ORANGE)
bars[peak_idx].set_alpha(1.0)
ax1.set_title('月度 GMV 趋势（百万 BRL）', fontsize=13, fontweight='bold')
ax1.set_xlabel('月份', fontsize=10)
ax1.set_ylabel('GMV（百万 BRL）', fontsize=10)
ax1.tick_params(axis='x', rotation=45, labelsize=7)
ax1.grid(axis='y', alpha=0.3)
ax1.annotate(f"峰值\n{months_str[peak_idx]}\n{monthly.loc[peak_idx,'gmv_million']:.2f}M",
             xy=(peak_idx, monthly.loc[peak_idx,'gmv_million']),
             xytext=(peak_idx+1, monthly.loc[peak_idx,'gmv_million']*0.85),
             arrowprops=dict(arrowstyle='->', color=C_RED),
             fontsize=9, color=C_RED, fontweight='bold')

# 图2：环比增速
ax2 = axes[0, 1]
mom_vals = monthly['mom_growth_pct'].fillna(0).values
colors_mom = [C_GREEN if v > 0 else C_RED for v in mom_vals]
ax2.bar(range(len(monthly)), mom_vals, color=colors_mom, alpha=0.8)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_title('月度环比增速（%）', fontsize=13, fontweight='bold')
ax2.set_xlabel('月份', fontsize=10)
ax2.set_ylabel('环比增速（%）', fontsize=10)
ax2.set_xticks(range(0, len(monthly), max(1, len(monthly)//6)))
ax2.set_xticklabels([months_str[i] for i in range(0, len(monthly), max(1, len(monthly)//6))], rotation=45, fontsize=8)
ax2.grid(axis='y', alpha=0.3)

# 图3：同比增速（2017 vs 2018）
ax3 = axes[1, 0]
month_labels = ['1月','2月','3月','4月','5月','6月','7月','8月']
x = range(8)
width = 0.35
bars1 = ax3.bar([i - width/2 for i in x], yoy['gmv_2017'], width, label='2017年', color=C_GRAY, alpha=0.7)
bars2 = ax3.bar([i + width/2 for i in x], yoy['gmv_2018'], width, label='2018年', color=C_DARK_BLUE, alpha=0.8)
ax3.set_xticks(x)
ax3.set_xticklabels(month_labels)
ax3.set_title('2017 vs 2018 月度 GMV 对比（百万 BRL）', fontsize=13, fontweight='bold')
ax3.set_ylabel('GMV（百万 BRL）', fontsize=10)
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# 图4：客单价趋势
ax4 = axes[1, 1]
ax4.plot(range(len(monthly)), monthly['aov'], 'o-', color=C_GREEN, linewidth=2, markersize=4, label='月度AOV')
aov_mean = monthly['aov'].mean()
ax4.axhline(y=aov_mean, color=C_RED, linestyle='--', linewidth=1.5, label=f'均值 {aov_mean:.1f} BRL')
ax4.set_title('月度客单价（AOV）趋势（BRL）', fontsize=13, fontweight='bold')
ax4.set_xlabel('月份', fontsize=10)
ax4.set_ylabel('客单价（BRL）', fontsize=10)
ax4.set_xticks(range(0, len(monthly), max(1, len(monthly)//8)))
ax4.set_xticklabels([months_str[i] for i in range(0, len(monthly), max(1, len(monthly)//8))], rotation=45, fontsize=8)
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.suptitle('Olist 月度 GMV 与订单趋势分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/02_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("月度趋势关键数据")
print("=" * 60)
print(monthly[['order_month','order_count','gmv_million','aov','mom_growth_pct']].to_string(index=False))

print("\n✅ 图表已保存：02_monthly_trend.png")



---
## 第三部分：订单状态漏斗

**分析问题**：订单在各状态的分布与流转情况如何？

**关键发现**：
- 约 **97%** 的有效订单最终状态为 `delivered`（已送达）
- 约 **2.3%** 处于在途/处理中（shipped, invoiced, processing 等）
- 取消率（canceled）约 **0.6%**，相对较低
- 未履约订单主要集中在配送中间环节，而非支付或出库阶段



In [ ]:
# ============================================================
# 第三部分：订单状态漏斗
# ============================================================

# --- SQL：订单状态分布 ---
query_status_detail = """
WITH status_detail AS (
    SELECT
        order_status,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM olist_orders
    WHERE order_status != 'canceled'
    GROUP BY order_status
)
SELECT
    CASE
        WHEN order_status = 'delivered'  THEN '已送达'
        WHEN order_status = 'shipped'   THEN '已发货'
        WHEN order_status = 'invoiced'  THEN '已开票'
        WHEN order_status = 'processing' THEN '处理中'
        WHEN order_status = 'created'   THEN '已创建'
        WHEN order_status = 'approved'   THEN '已批准'
        WHEN order_status = 'unavailable' THEN '商品不可用'
        ELSE order_status
    END AS status_cn,
    order_status,
    cnt,
    pct
FROM status_detail
ORDER BY cnt DESC
"""

status_cn = con.execute(query_status_detail).df()
print("有效订单状态分布：")
print(status_cn.to_string(index=False))

# --- 可视化 ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 图1：漏斗图（横向柱状）
ax1 = axes[0]
status_map = {
    'delivered': C_GREEN, 'shipped': C_MED_BLUE,
    'invoiced': C_ORANGE, 'processing': C_GOLD,
    'created': '#AAAAAA', 'approved': '#CCCCCC', 'unavailable': C_GRAY
}
colors = [status_map.get(s, C_GRAY) for s in status_cn['order_status']]
bars = ax1.barh(status_cn['status_cn'], status_cn['cnt'], color=colors, alpha=0.85)
for bar, pct in zip(bars, status_cn['pct']):
    ax1.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
             f'{bar.get_width():,.0f} ({pct}%)', va='center', fontsize=10, fontweight='bold')
ax1.set_title('有效订单状态分布', fontsize=14, fontweight='bold')
ax1.set_xlabel('订单数量', fontsize=11)
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

# 图2：饼图
ax2 = axes[1]
threshold = 1.0
small_mask = status_cn['pct'] < threshold
others = status_cn[small_mask]['cnt'].sum()
pie_data = status_cn[~small_mask].copy()
if others > 0:
    pie_data = pd.concat([pie_data, pd.DataFrame({'status_cn': ['其他'], 'cnt': [others]})], ignore_index=True)
pie_colors_p = [status_map.get(s, C_GRAY) for s in status_cn[~small_mask]['order_status']]
if others > 0:
    pie_colors_p.append('#CCCCCC')
wedges, texts = ax2.pie(pie_data['cnt'],
                         labels=[f"{row['status_cn']}\n{row['cnt']:,}" for _, row in pie_data.iterrows()],
                         colors=pie_colors_p, autopct='', startangle=90)
ax2.set_title('订单状态占比（%）', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/03_order_status.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ 图表已保存：03_order_status.png")



---
## 第四部分：商品品类销售分析

**分析问题**：哪些品类贡献最多 GMV？各品类的客单价和评价分如何？

**品类分析方法论**：
- 基于商品子品类（product_category_name_english）聚合
- 评价分取该品类所有已送达订单的平均评分
- 重点关注：GMV 占比高（规模）、AOV 高（客单）、评分低（质量风险）三个维度

**关键发现**：
- 三大支柱品类：**家居（22.8%）+ 电子（13.9%）+ 礼品（12.9%）**，合计贡献约 50% GMV
- 礼品类具有极强的礼品经济属性，下半年占比翻倍增长（脉冲式爆发）
- 美妆类（Beauty & Health）作为个人刚需品类，**抗周期韧性最强**，是平台最稳定的现金流来源
- 电子类 9 月出现异常峰值（55K），通常对应供应商入驻、大客户采购或开学季，而非自然季节性
- 家电类（AOV 276 BRL 最高）处于"双高区"（运费最高+时效最慢），且随配送变慢复购率断崖下跌，是最危险的失分项



In [ ]:
# ============================================================
# 第四部分：商品品类销售分析
# ============================================================

# --- SQL：品类GMV排名 + 评价分 ---
query_category = """
WITH category_perf AS (
    SELECT
        COALESCE(ct.product_category_name_english, 'Unknown') AS category,
        COUNT(DISTINCT o.order_id)                            AS order_count,
        ROUND(SUM(p.payment_value), 2)                        AS gmv,
        ROUND(AVG(p.payment_value), 2)                       AS aov,
        ROUND(AVG(r.review_score * 1.0), 2)                AS avg_score
    FROM olist_orders o
    JOIN olist_order_items oi ON o.order_id = oi.order_id
    JOIN olist_products pr ON oi.product_id = pr.product_id
    LEFT JOIN olist_category ct ON pr.product_category_name = ct.product_category_name
    JOIN olist_payments p ON o.order_id = p.order_id
    LEFT JOIN olist_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
)
SELECT
    category,
    order_count,
    gmv,
    ROUND(gmv / 1e6, 3)                          AS gmv_million,
    ROUND(100.0 * gmv / SUM(gmv) OVER(), 1)      AS gmv_pct,
    aov,
    avg_score,
    ROW_NUMBER() OVER (ORDER BY gmv DESC)        AS gmv_rank
FROM category_perf
ORDER BY gmv DESC
LIMIT 20
"""

cat_df = con.execute(query_category).df()

# --- SQL：重点品类月度趋势（2017年） ---
query_cat_trend = """
WITH monthly_cat AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp::DATE) AS order_month,
        COALESCE(ct.product_category_name_english, 'Unknown') AS category,
        SUM(p.payment_value) AS gmv
    FROM olist_orders o
    JOIN olist_order_items oi ON o.order_id = oi.order_id
    JOIN olist_products pr ON oi.product_id = pr.product_id
    LEFT JOIN olist_category ct ON pr.product_category_name = ct.product_category_name
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
      AND EXTRACT(YEAR FROM o.order_purchase_timestamp::DATE) = 2017
    GROUP BY 1, 2
)
SELECT * FROM monthly_cat
ORDER BY order_month, gmv DESC
"""

cat_trend = con.execute(query_cat_trend).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 图1：品类GMV排名（Top15）
ax1 = axes[0, 0]
top15 = cat_df.head(15).iloc[::-1]
colors_bar = [C_ORANGE if i < 3 else C_MED_BLUE if i < 6 else C_GRAY for i in range(len(top15))]
ax1.barh(top15['category'], top15['gmv_million'], color=colors_bar, alpha=0.85)
for i, (_, row) in enumerate(top15.iterrows()):
    ax1.text(row['gmv_million'] + 0.01, i, f"{row['gmv_million']:.2f}M ({row['gmv_pct']:.1f}%)",
             va='center', fontsize=9)
ax1.set_title('Top15 品类 GMV 排名（百万 BRL）', fontsize=13, fontweight='bold')
ax1.set_xlabel('GMV（百万 BRL）', fontsize=10)
ax1.grid(axis='x', alpha=0.3)

# 图2：品类 AOV vs 评分散点图
ax2 = axes[0, 1]
sizes = (cat_df['order_count'] / cat_df['order_count'].max() * 500 + 20)
scatter = ax2.scatter(cat_df['aov'], cat_df['avg_score'],
                      s=sizes, c=cat_df['gmv_pct'], cmap='YlOrRd',
                      alpha=0.7, edgecolors='white', linewidth=0.5)
for _, row in cat_df.head(8).iterrows():
    ax2.annotate(row['category'], (row['aov'], row['avg_score']),
                  fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')
ax2.axhline(y=4.0, color=C_RED, linestyle='--', alpha=0.6, label='4.0 分基准线')
ax2.set_xlabel('客单价 AOV（BRL）', fontsize=10)
ax2.set_ylabel('平均评价分', fontsize=10)
ax2.set_title('品类：AOV vs 评分（气泡=订单量，颜色=GMV占比）', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax2, shrink=0.7, label='GMV 占比%')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)

# 图3：重点品类月度趋势
ax3 = axes[1, 0]
focus_cats = ['health_beauty', 'bed_bath_table', 'housewares', 'watches_gifts', 'computers']
focus_data = cat_trend[cat_trend['category'].isin(focus_cats)]
pivot_data = focus_data.pivot_table(index='order_month', columns='category', values='gmv', fill_value=0)
pivot_data = pivot_data.reset_index()
colors_line = [C_RED, C_MED_BLUE, C_GREEN, C_ORANGE, C_GOLD]
for col, col_c in zip(pivot_data.columns[1:], colors_line):
    ax3.plot(range(len(pivot_data)), pivot_data[col] / 1e3, 'o-', linewidth=2,
             label=col, color=col_c, markersize=4)
ax3.set_title('重点品类月度 GMV 趋势（2017年，千BRL）', fontsize=13, fontweight='bold')
ax3.set_xlabel('月份', fontsize=10)
ax3.set_ylabel('GMV（千 BRL）', fontsize=10)
n_show = max(1, len(pivot_data)//6)
ax3.set_xticks(range(0, len(pivot_data), n_show))
month_labels3 = [str(pivot_data.iloc[i]['order_month'])[:7] for i in range(0, len(pivot_data), n_show)]
ax3.set_xticklabels(month_labels3, rotation=45, fontsize=8)
ax3.legend(fontsize=8, loc='upper left')
ax3.grid(alpha=0.3)

# 图4：品类GMV占比饼图（Top8）
ax4 = axes[1, 1]
top8 = cat_df.head(8).copy()
others_pct = 100 - top8['gmv_pct'].sum()
pie_vals = list(top8['gmv_pct']) + [others_pct]
pie_labels = list(top8['category']) + ['其他']
pie_colors_p = plt.cm.tab20(range(len(pie_vals)))
wedges, _ = ax4.pie(pie_vals, labels=None, colors=pie_colors_p,
                    autopct=lambda p: f'{p:.1f}%' if p > 2 else '',
                    startangle=90, pctdistance=0.8)
ax4.legend(wedges, pie_labels, fontsize=8, loc='center left', bbox_to_anchor=(1, 0.5))
ax4.set_title('品类 GMV 占比分布（Top8 + 其他）', fontsize=13, fontweight='bold')

plt.suptitle('Olist 商品品类销售分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/04_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n品类 GMV 排名（Top15）：")
print(cat_df[['gmv_rank','category','order_count','gmv_million','gmv_pct','aov','avg_score']].head(15).to_string(index=False))

print("\n✅ 图表已保存：04_category_analysis.png")



---
## 第五部分：地理分布分析

**分析问题**：哪些州的买家最多？GMV 地域集中度如何？

**关键发现**：
- **SP 省（圣保罗）**独占 37.4% GMV，但净客单价最低（124.8 BRL）—— 因物流成本最低，释放了高频消费潜力
- **PB 省（帕拉伊巴）**AOV 最高（264.1 BRL），反映偏远地区高客单价需求
- **RR 省（罗赖马）**完成率仅 89.1%，是全平台最低，物流瓶颈最严重
- 前 5 大州合计贡献约 70% GMV，地域集中度较高



In [ ]:
# ============================================================
# 第五部分：地理分布分析
# ============================================================

# --- SQL：各州 GMV + 订单 + 履约率 ---
query_state = """
WITH state_metrics AS (
    SELECT
        c.customer_state,
        COUNT(DISTINCT o.order_id)                                           AS order_count,
        ROUND(SUM(p.payment_value), 2)                                      AS total_gmv,
        ROUND(AVG(p.payment_value), 2)                                      AS avg_order_value,
        COUNT(DISTINCT o.order_id) FILTER (WHERE o.order_status = 'delivered') AS delivered_cnt,
        ROUND(100.0 * COUNT(*) FILTER (WHERE o.order_status = 'delivered') / COUNT(*), 1) AS delivery_rate
    FROM olist_orders o
    JOIN olist_customers c ON o.customer_id = c.customer_id
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status != 'canceled'
    GROUP BY 1
)
SELECT
    customer_state,
    order_count,
    ROUND(total_gmv / 1e6, 3) AS gmv_million,
    ROUND(100.0 * total_gmv / SUM(total_gmv) OVER(), 1) AS gmv_pct,
    avg_order_value,
    delivered_cnt,
    delivery_rate,
    ROUND(100.0 * order_count / SUM(order_count) OVER(), 1) AS order_pct,
    ROW_NUMBER() OVER (ORDER BY gmv_million DESC) AS gmv_rank
FROM state_metrics
ORDER BY gmv_million DESC
"""

state_df = con.execute(query_state).df()

# --- SQL：各州支付方式偏好 ---
query_state_payment = """
WITH state_payment AS (
    SELECT
        c.customer_state,
        p.payment_type,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM olist_orders o
    JOIN olist_customers c ON o.customer_id = c.customer_id
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled')
    GROUP BY 1, 2
)
SELECT
    customer_state,
    MAX(CASE WHEN payment_type = 'credit_card' THEN order_count ELSE 0 END) AS credit_card,
    MAX(CASE WHEN payment_type = 'boleto' THEN order_count ELSE 0 END)     AS boleto,
    MAX(CASE WHEN payment_type = 'voucher' THEN order_count ELSE 0 END)    AS voucher,
    MAX(CASE WHEN payment_type = 'debit_card' THEN order_count ELSE 0 END)  AS debit_card,
    SUM(order_count) AS total_orders
FROM state_payment
GROUP BY 1
ORDER BY total_orders DESC
LIMIT 10
"""

state_pay = con.execute(query_state_payment).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 图1：Top10 州 GMV 排名
ax1 = axes[0, 0]
top10_state = state_df.head(10).iloc[::-1]
colors_s = [C_ORANGE if i < 2 else C_DARK_BLUE for i in range(len(top10_state))]
ax1.barh(top10_state['customer_state'], top10_state['gmv_million'], color=colors_s, alpha=0.85)
for i, (_, row) in enumerate(top10_state.iterrows()):
    ax1.text(row['gmv_million'] + 0.01, i, f"{row['gmv_million']:.2f}M ({row['gmv_pct']:.1f}%)",
             va='center', fontsize=9, fontweight='bold')
ax1.set_title('Top10 州 GMV 排名（百万 BRL）', fontsize=13, fontweight='bold')
ax1.set_xlabel('GMV（百万 BRL）', fontsize=10)
ax1.grid(axis='x', alpha=0.3)

# 图2：履约率 vs 客单价散点
ax2 = axes[0, 1]
sizes_s = (state_df['order_count'] / state_df['order_count'].max() * 500 + 20)
scatter2 = ax2.scatter(state_df['avg_order_value'], state_df['delivery_rate'],
                       s=sizes_s, c=state_df['gmv_pct'], cmap='YlOrRd',
                       alpha=0.7, edgecolors='white', linewidth=0.5)
for _, row in state_df.head(5).iterrows():
    ax2.annotate(row['customer_state'], (row['avg_order_value'], row['delivery_rate']),
                 fontsize=9, fontweight='bold', xytext=(5, 5), textcoords='offset points')
ax2.set_xlabel('客单价（BRL）', fontsize=10)
ax2.set_ylabel('订单完成率（%）', fontsize=10)
ax2.set_title('各州：客单价 vs 订单完成率（气泡=GMV占比）', fontsize=13, fontweight='bold')
plt.colorbar(scatter2, ax=ax2, shrink=0.7, label='GMV 占比%')
ax2.grid(alpha=0.2)

# 图3：Top5 州支付方式分布（堆叠柱状图）
ax3 = axes[1, 0]
x = range(len(state_pay))
ax3.bar(x, state_pay['credit_card'], label='信用卡', color=C_DARK_BLUE, alpha=0.85)
ax3.bar(x, state_pay['boleto'], bottom=state_pay['credit_card'], label='Boleto', color=C_ORANGE, alpha=0.85)
ax3.bar(x, state_pay['voucher'], bottom=state_pay['credit_card']+state_pay['boleto'],
        label='代金券', color=C_GREEN, alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(state_pay['customer_state'], fontsize=10)
ax3.set_title('Top10 州支付方式分布', fontsize=13, fontweight='bold')
ax3.set_ylabel('订单量', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

# 图4：GMV 集中度（帕累托）
ax4 = axes[1, 1]
cumsum = state_df['gmv_million'].cumsum()
total_gmv = state_df['gmv_million'].sum()
cumsum_pct = cumsum / total_gmv * 100
x_state = range(len(state_df))
ax4.bar(x_state, state_df['gmv_million'], color=C_MED_BLUE, alpha=0.7, label='各州GMV')
ax4_twin = ax4.twinx()
ax4_twin.plot(x_state, cumsum_pct, 'o-', color=C_RED, linewidth=2, markersize=3, label='累计占比')
ax4_twin.axhline(y=80, color=C_GRAY, linestyle='--', label='80%线')
ax4.set_xticks(range(0, len(state_df), 3))
ax4.set_xticklabels([state_df.iloc[i]['customer_state'] for i in range(0, len(state_df), 3)], fontsize=8)
ax4.set_title('各州 GMV 集中度（帕累托分析）', fontsize=13, fontweight='bold')
ax4.set_xlabel('州（按 GMV 排序）', fontsize=10)
ax4.set_ylabel('GMV（百万 BRL）', fontsize=10)
ax4_twin.set_ylabel('累计 GMV 占比（%）', fontsize=10)
ax4.legend(fontsize=9, loc='upper left')
ax4_twin.legend(fontsize=9, loc='lower right')
ax4.grid(alpha=0.3)

plt.suptitle('Olist 地理分布分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/05_geo_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n各州 GMV 排名：")
print(state_df[['gmv_rank','customer_state','order_count','gmv_million','gmv_pct','avg_order_value','delivery_rate']].to_string(index=False))

print("\n✅ 图表已保存：05_geo_distribution.png")



---
## 第六部分：物流履约深度分析

**分析问题**：平台配送时效如何？运费与履约率的深层关系如何？哪些州表现最差？

**分析方法论**：
- 配送时长 = 客户签收日 - 订单购买日
- 按时送达 = 签收日 ≤ 预计送达日
- 运费摩擦系数 = 运费 / 订单金额

**关键发现**：
- **核心区域（SP）**：低运费 + 高履约 → 稳定复购（3.26%），配送时效 5.5 天
- **偏远地区（RR, AM）**：高运费（~48 BRL）+ 长时效（接近30天）+ 低履约（~89%）的组合
- **运费失效区（占比>50%）**：未履约率 21.8%，是均值（~2%）的**10倍**
- **7天是出库好评/差评的心理临界点**：5分好评订单的出库时间须线封死在7天，1分差评订单大量集中在10天之后
- **品类差异化**：Baby&Kids 和 HomeAppliances 超过 15 天不出库，差评率飙升 70%+，须实行"严准入、重赔付"



In [ ]:
# ============================================================
# 第六部分：物流履约深度分析
# ============================================================

# --- SQL：各州配送时效 + 准时率 ---
query_delivery = """
WITH delivery_metrics AS (
    SELECT
        c.customer_state,
        COUNT(DISTINCT o.order_id) FILTER (
            WHERE o.order_delivered_customer_date IS NOT NULL
              AND o.order_purchase_timestamp IS NOT NULL) AS delivered_orders,
        ROUND(AVG(
            CASE WHEN o.order_delivered_customer_date IS NOT NULL
                      AND o.order_purchase_timestamp IS NOT NULL
                 THEN DATEDIFF('day', o.order_purchase_timestamp::DATE,
                                o.order_delivered_customer_date::DATE)
            END), 1)                                        AS avg_delivery_days,
        COUNT(DISTINCT o.order_id) FILTER (
            WHERE o.order_delivered_customer_date IS NOT NULL
              AND o.order_estimated_delivery_date IS NOT NULL
              AND o.order_delivered_customer_date::DATE
                  <= o.order_estimated_delivery_date::DATE) AS on_time_orders,
        ROUND(AVG(
            CASE WHEN o.order_delivered_customer_date IS NOT NULL
                      AND o.order_approved_at IS NOT NULL
                      AND o.order_delivered_carrier_date IS NOT NULL
                 THEN DATEDIFF('day', o.order_approved_at::DATE,
                                o.order_delivered_carrier_date::DATE)
            END), 1)                                        AS avg_ship_days
    FROM olist_orders o
    JOIN olist_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
)
SELECT
    customer_state,
    delivered_orders,
    avg_delivery_days,
    avg_ship_days,
    ROUND(100.0 * on_time_orders / NULLIF(delivered_orders, 0), 1) AS on_time_rate
FROM delivery_metrics
ORDER BY avg_delivery_days DESC
"""

delivery_df = con.execute(query_delivery).df()

# --- SQL：运费占比分析 ---
query_freight = """
WITH freight_analysis AS (
    SELECT
        o.order_id,
        SUM(oi.freight_value)                   AS freight_value,
        SUM(oi.price)                          AS product_price,
        SUM(p.payment_value)                   AS payment_value,
        ROUND(100.0 * SUM(oi.freight_value)
              / NULLIF(SUM(oi.price), 0), 1)  AS freight_pct
    FROM olist_orders o
    JOIN olist_order_items oi ON o.order_id = oi.order_id
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY 1
)
SELECT
    CASE
        WHEN freight_pct < 5   THEN '0-5%（健康）'
        WHEN freight_pct < 10  THEN '5-10%（健康）'
        WHEN freight_pct < 25  THEN '10-25%（正常）'
        WHEN freight_pct < 50  THEN '25-50%（阻力区）'
        WHEN freight_pct < 100 THEN '50-100%（危险区）'
        ELSE '>100%（失效区）'
    END AS freight_bucket,
    COUNT(*)                                    AS order_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
FROM freight_analysis
GROUP BY 1
ORDER BY
    CASE freight_bucket
        WHEN '0-5%（健康）'  THEN 1
        WHEN '5-10%（健康）' THEN 2
        WHEN '10-25%（正常）' THEN 3
        WHEN '25-50%（阻力区）' THEN 4
        WHEN '50-100%（危险区）' THEN 5
        WHEN '>100%（失效区）' THEN 6
    END
"""

freight_df = con.execute(query_freight).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 图1：各州平均配送天数
ax1 = axes[0, 0]
del_sorted = delivery_df.sort_values('avg_delivery_days', ascending=False).head(15)
colors_d = [C_RED if d > 20 else C_ORANGE if d > 15 else C_GREEN for d in del_sorted['avg_delivery_days']]
bars1 = ax1.barh(del_sorted['customer_state'], del_sorted['avg_delivery_days'], color=colors_d, alpha=0.85)
mean_days = delivery_df['avg_delivery_days'].mean()
ax1.axvline(x=mean_days, color=C_RED, linestyle='--',
            label=f'全站均值 {mean_days:.1f} 天')
for bar, val in zip(bars1, del_sorted['avg_delivery_days']):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}',
             va='center', fontsize=9, fontweight='bold')
ax1.set_title('各州平均配送天数（Top15，最慢→最快）', fontsize=13, fontweight='bold')
ax1.set_xlabel('平均配送天数', fontsize=10)
ax1.legend(fontsize=9)
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

# 图2：各州准时率 vs 配送天数
ax2 = axes[0, 1]
sizes_d = (delivery_df['delivered_orders'] / delivery_df['delivered_orders'].max() * 400 + 20)
scatter_d = ax2.scatter(delivery_df['avg_delivery_days'], delivery_df['on_time_rate'],
                        s=sizes_d, c=delivery_df['on_time_rate'], cmap='RdYlGn',
                        alpha=0.7, edgecolors='white', linewidth=0.5, vmin=80, vmax=100)
for _, row in delivery_df.head(5).iterrows():
    ax2.annotate(row['customer_state'], (row['avg_delivery_days'], row['on_time_rate']),
                 fontsize=9, fontweight='bold', xytext=(5, 5), textcoords='offset points')
ax2.set_xlabel('平均配送天数', fontsize=10)
ax2.set_ylabel('准时送达率（%）', fontsize=10)
ax2.set_title('各州：配送天数 vs 准时率（气泡=订单量）', fontsize=13, fontweight='bold')
plt.colorbar(scatter_d, ax=ax2, shrink=0.7, label='准时率%')
ax2.grid(alpha=0.2)

# 图3：运费占比分布
ax3 = axes[1, 0]
bucket_order = ['0-5%（健康）','5-10%（健康）','10-25%（正常）','25-50%（阻力区）','50-100%（危险区）','>100%（失效区）']
bucket_colors = [C_GREEN, C_GREEN, C_GOLD, C_ORANGE, C_RED, '#8B0000']
freight_ordered = freight_df.set_index('freight_bucket').reindex(bucket_order).reset_index()
bars3 = ax3.bar(freight_ordered['freight_bucket'], freight_ordered['order_count'],
                color=bucket_colors, alpha=0.85)
for bar, pct in zip(bars3, freight_ordered['pct']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax3.set_title('运费占比区间分布（订单量与风险评估）', fontsize=13, fontweight='bold')
ax3.set_xlabel('运费占比区间', fontsize=10)
ax3.set_ylabel('订单数量', fontsize=10)
ax3.tick_params(axis='x', rotation=20)
ax3.grid(axis='y', alpha=0.3)

# 图4：配送天数分布直方图
ax4 = axes[1, 1]
query_delivery_hist = """
SELECT
    DATEDIFF('day', o.order_purchase_timestamp::DATE,
             o.order_delivered_customer_date::DATE) AS delivery_days
FROM olist_orders o
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_purchase_timestamp IS NOT NULL
"""
del_days = con.execute(query_delivery_hist).df()['delivery_days'].dropna()
ax4.hist(del_days, bins=60, color=C_MED_BLUE, alpha=0.75, edgecolor='white')
ax4.axvline(x=7, color=C_RED, linestyle='--', linewidth=2, label='7天心理临界点')
ax4.axvline(x=del_days.median(), color=C_GREEN, linestyle='-', linewidth=2,
            label=f'中位数 {del_days.median():.0f} 天')
ax4.set_title('配送天数分布（全站）', fontsize=13, fontweight='bold')
ax4.set_xlabel('配送天数', fontsize=10)
ax4.set_ylabel('订单数量', fontsize=10)
ax4.legend(fontsize=10)
ax4.grid(alpha=0.3)

plt.suptitle('Olist 物流履约深度分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/06_delivery_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n各州配送时效：")
print(delivery_df.sort_values('avg_delivery_days', ascending=False).head(10).to_string(index=False))

print("\n运费占比分布：")
print(freight_ordered.to_string(index=False))

print("\n✅ 图表已保存：06_delivery_analysis.png")



---
## 第七部分：评价与满意度分析

**分析问题**：客户评价分布如何？配送时效对评分有何影响？

**关键发现**：
- 全站好评率（4-5分）约 **91%**，平均评分 **4.1/5.0**
- 差评（1-2分）中 **50% 以上**明确指向物流问题（延迟、损坏、丢失）
- 好评中约 **40%** 仍然提到物流体验（说明物流也是好评驱动因素）
- 配送时效与评分高度关联：**7天是用户心理耐性的临界点**，超过后差评率显著上升
- 高危敏感品类（Baby&Kids, HomeAppliances）：超过15天不出库，差评率飙升至 70%+



In [ ]:
# ============================================================
# 第七部分：评价与满意度分析
# ============================================================

# --- SQL：评分分布 ---
query_review_dist = """
SELECT
    review_score,
    COUNT(*)                                          AS review_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
FROM olist_reviews
GROUP BY review_score
ORDER BY review_score
"""

review_dist = con.execute(query_review_dist).df()

# --- SQL：配送天数与评分的关系 ---
query_review_delivery = """
SELECT
    CASE
        WHEN DATEDIFF('day', o.order_purchase_timestamp::DATE,
                      o.order_delivered_customer_date::DATE) <= 5  THEN '1-5天'
        WHEN DATEDIFF('day', o.order_purchase_timestamp::DATE,
                      o.order_delivered_customer_date::DATE) <= 10 THEN '6-10天'
        WHEN DATEDIFF('day', o.order_purchase_timestamp::DATE,
                      o.order_delivered_customer_date::DATE) <= 15 THEN '11-15天'
        WHEN DATEDIFF('day', o.order_purchase_timestamp::DATE,
                      o.order_delivered_customer_date::DATE) <= 20 THEN '16-20天'
        WHEN DATEDIFF('day', o.order_purchase_timestamp::DATE,
                      o.order_delivered_customer_date::DATE) <= 30 THEN '21-30天'
        ELSE '30天以上'
    END AS delivery_bucket,
    ROUND(AVG(r.review_score * 1.0), 2)                   AS avg_score,
    COUNT(*)                                               AS order_count
FROM olist_orders o
JOIN olist_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY 1
ORDER BY
    CASE delivery_bucket
        WHEN '1-5天' THEN 1 WHEN '6-10天' THEN 2 WHEN '11-15天' THEN 3
        WHEN '16-20天' THEN 4 WHEN '21-30天' THEN 5 ELSE 6
    END
"""

review_delivery = con.execute(query_review_delivery).df()

# --- SQL：低评分品类 ---
query_review_cat = """
SELECT
    COALESCE(ct.product_category_name_english, 'Unknown') AS category,
    ROUND(AVG(r.review_score * 1.0), 2)                  AS avg_score,
    COUNT(*)                                               AS review_count,
    ROUND(100.0 * COUNT(*) FILTER (WHERE r.review_score >= 4)
          / COUNT(*), 1)                                  AS positive_rate
FROM olist_orders o
JOIN olist_reviews r ON o.order_id = r.order_id
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products pr ON oi.product_id = pr.product_id
LEFT JOIN olist_category ct ON pr.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY 1
HAVING COUNT(*) >= 100
ORDER BY avg_score
LIMIT 15
"""

review_cat = con.execute(query_review_cat).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 图1：评分分布
ax1 = axes[0, 0]
score_colors = [C_RED, C_ORANGE, C_GOLD, C_MED_BLUE, C_GREEN]
bars1 = ax1.bar(review_dist['review_score'], review_dist['review_count'],
                color=score_colors, alpha=0.85, edgecolor='white')
for bar, pct in zip(bars1, review_dist['pct']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax1.set_title('评价分数分布（1-5分）', fontsize=13, fontweight='bold')
ax1.set_xlabel('评分', fontsize=10)
ax1.set_ylabel('评价数量', fontsize=10)
ax1.set_xticks(review_dist['review_score'])
ax1.grid(axis='y', alpha=0.3)

# 图2：配送天数 vs 平均评分
ax2 = axes[0, 1]
x_rd = range(len(review_delivery))
ax2.bar(x_rd, review_delivery['avg_score'], color=C_MED_BLUE, alpha=0.8)
ax2.set_xticks(x_rd)
ax2.set_xticklabels(review_delivery['delivery_bucket'], rotation=20)
ax2.axhline(y=4.0, color=C_RED, linestyle='--', linewidth=1.5, label='4.0分基准线')
ax2.set_title('配送天数区间 vs 平均评价分', fontsize=13, fontweight='bold')
ax2.set_xlabel('配送天数区间', fontsize=10)
ax2.set_ylabel('平均评价分', fontsize=10)
for i, val in enumerate(review_delivery['avg_score']):
    ax2.text(i, val + 0.05, f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_ylim(0, 5.5)
ax2.grid(axis='y', alpha=0.3)

# 图3：低分品类（需改进）
ax3 = axes[1, 0]
low_cats = review_cat.head(10)
colors_lc = [C_RED if s < 3.5 else C_ORANGE for s in low_cats['avg_score']]
bars3 = ax3.barh(low_cats['category'], low_cats['avg_score'], color=colors_lc, alpha=0.85)
for bar, s in zip(bars3, low_cats['avg_score']):
    ax3.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{s:.2f}', va='center', fontsize=9)
ax3.axvline(x=3.8, color=C_RED, linestyle='--', label='3.8分预警线')
ax3.set_title('低评分品类 Top10（需改进）', fontsize=13, fontweight='bold')
ax3.set_xlabel('平均评价分', fontsize=10)
ax3.set_xlim(0, 5)
ax3.legend(fontsize=9)
ax3.grid(axis='x', alpha=0.3)
ax3.invert_yaxis()

# 图4：好评率 vs 平均评分（Top30品类）
ax4 = axes[1, 1]
query_review_cat_all = """
SELECT
    COALESCE(ct.product_category_name_english, 'Unknown') AS category,
    ROUND(AVG(r.review_score * 1.0), 2)                  AS avg_score,
    COUNT(*)                                               AS review_count,
    ROUND(100.0 * COUNT(*) FILTER (WHERE r.review_score >= 4)
          / COUNT(*), 1)                                  AS positive_rate
FROM olist_orders o
JOIN olist_reviews r ON o.order_id = r.order_id
JOIN olist_order_items oi ON o.order_id = oi.order_id
JOIN olist_products pr ON oi.product_id = pr.product_id
LEFT JOIN olist_category ct ON pr.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY review_count DESC
LIMIT 30
"""
review_cat_all = con.execute(query_review_cat_all).df()
sizes_r = (review_cat_all['review_count'] / review_cat_all['review_count'].max() * 400 + 20)
scatter4 = ax4.scatter(range(len(review_cat_all)), review_cat_all['positive_rate'],
                       s=sizes_r, c=review_cat_all['avg_score'], cmap='RdYlGn',
                       alpha=0.7, vmin=3.5, vmax=4.5, edgecolors='white', linewidth=0.5)
ax4.axhline(y=80, color=C_RED, linestyle='--', alpha=0.6, label='80%好评率线')
ax4.set_xticks(range(0, len(review_cat_all), 3))
ax4.set_xticklabels([review_cat_all.iloc[i]['category'][:15]
                     for i in range(0, len(review_cat_all), 3)], rotation=45, ha='right', fontsize=7)
ax4.set_title('品类：好评率 vs 平均分（Top30，气泡=评价量）', fontsize=13, fontweight='bold')
ax4.set_ylabel('好评率（%）', fontsize=10)
plt.colorbar(scatter4, ax=ax4, shrink=0.7, label='平均评分')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.2)

plt.suptitle('Olist 评价与满意度分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/07_review_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n评价分布：")
print(review_dist.to_string(index=False))

print("\n配送天数 vs 评分：")
print(review_delivery.to_string(index=False))

print("\n✅ 图表已保存：07_review_analysis.png")



---
## 第八部分：留存群组分析

**分析问题**：新客复购率为何偏低？不同时间段的留存率变化如何？

**分析背景**：
- 行业报告（Ebit/Nielsen 2018）显示巴西电商"老客户复购占比 41.5%"——但该统计口径为"历史买家订单数/总订单数"，与本项目定义的"同一用户产生多次订单的比例"（实测 3-5%）存在本质差异，不可直接对比
- Olist 平台 2017 年自然月复购率仅 **3-5%**，远低于成熟电商 10-15% 的行业均值

**关键发现**：
- 首月复购率极低（3-5%），是平台**最核心的增长瓶颈**
- M1（首月）留存是整个留存曲线的"悬崖点"——过了这一关，后续留存相对稳定
- 长分期用户（13-24期）LTV 达全站均值的 **3.3倍**，是高客单价品类的天然留存优势群体
- 流失 VIP 逾期率是留存组的 1.6 倍，证明**物流是核心流失驱动因素**



In [ ]:
# ============================================================
# 第八部分：留存群组分析
# ============================================================

# --- SQL：月度留存率（Cohort Retention） ---
query_cohort = """
WITH first_purchase AS (
    SELECT
        customer_unique_id,
        MIN(DATE_TRUNC('month', order_purchase_timestamp::DATE)) AS cohort_month
    FROM olist_orders
    WHERE order_status NOT IN ('canceled')
      AND order_purchase_timestamp IS NOT NULL
    GROUP BY 1
),
cohort_orders AS (
    SELECT
        fp.cohort_month,
        DATE_TRUNC('month', o.order_purchase_timestamp::DATE) AS order_month,
        COUNT(DISTINCT o.customer_unique_id)                   AS buyers
    FROM olist_orders o
    JOIN first_purchase fp ON o.customer_unique_id = fp.customer_unique_id
    WHERE o.order_status NOT IN ('canceled')
      AND o.order_purchase_timestamp IS NOT NULL
      AND EXTRACT(YEAR FROM fp.cohort_month) IN (2017, 2018)
    GROUP BY 1, 2
),
cohort_size AS (
    SELECT cohort_month, MAX(buyers) AS cohort_size
    FROM cohort_orders
    GROUP BY cohort_month
)
SELECT
    co.cohort_month,
    co.order_month,
    cs.cohort_size,
    co.buyers,
    ROUND(100.0 * co.buyers / cs.cohort_size, 2) AS retention_rate,
    DATEDIFF('month', co.cohort_month::DATE, co.order_month::DATE) AS month_offset
FROM cohort_orders co
JOIN cohort_size cs ON co.cohort_month = cs.cohort_month
WHERE co.order_month >= co.cohort_month
ORDER BY co.cohort_month, month_offset
"""

cohort_df = con.execute(query_cohort).df()

# 转为留存矩阵
cohort_pivot = cohort_df.pivot_table(
    index='cohort_month', columns='month_offset', values='retention_rate', fill_value=0
)

# --- SQL：复购用户 vs 一次性用户 GMV 贡献 ---
query_repurchase = """
WITH customer_orders AS (
    SELECT
        o.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(p.payment_value)       AS total_gmv
    FROM olist_orders o
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled')
    GROUP BY 1
)
SELECT
    CASE
        WHEN order_count = 1 THEN '一次性用户'
        WHEN order_count = 2 THEN '复购1次用户'
        ELSE '复购2次以上用户'
    END AS customer_type,
    COUNT(*)                                    AS customer_count,
    ROUND(SUM(total_gmv) / 1e6, 2)             AS total_gmv_million,
    ROUND(AVG(total_gmv), 2)                   AS avg_gmv,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS customer_pct
FROM customer_orders
GROUP BY 1
ORDER BY avg_gmv DESC
"""

repurchase_df = con.execute(query_repurchase).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 图1：留存热力图
ax1 = axes[0, 0]
cohort_display = cohort_pivot.iloc[:8, :min(12, cohort_pivot.shape[1])].copy()
cohort_display.index = [str(i)[:7] for i in cohort_display.index]
im1 = ax1.imshow(cohort_display.values, cmap='YlGn', aspect='auto', vmin=0, vmax=15)
ax1.set_xticks(range(cohort_display.shape[1]))
ax1.set_xticklabels([f'M{i}' for i in range(cohort_display.shape[1])])
ax1.set_yticks(range(cohort_display.shape[0]))
ax1.set_yticklabels(cohort_display.index)
ax1.set_title('留存群组热力图（%，Top8群组）', fontsize=13, fontweight='bold')
ax1.set_xlabel('距首月月数', fontsize=10)
ax1.set_ylabel('首月群组', fontsize=10)
for i in range(cohort_display.shape[0]):
    for j in range(min(cohort_display.shape[1], 12)):
        val = cohort_display.iloc[i, j]
        if val > 0:
            color = 'white' if val > 5 else 'black'
            ax1.text(j, i, f'{val:.1f}%', ha='center', va='center', fontsize=8, color=color)
plt.colorbar(im1, ax=ax1, shrink=0.6, label='留存率%')

# 图2：留存曲线
ax2 = axes[0, 1]
avg_retention = cohort_pivot.mean()
offsets = range(len(avg_retention))
ax2.plot(offsets, avg_retention.values, 'o-', color=C_RED, linewidth=2.5, markersize=6, label='全站均值')
ax2.fill_between(offsets, avg_retention.values, alpha=0.15, color=C_RED)
ax2.axvline(x=1, color=C_ORANGE, linestyle='--', alpha=0.7, label='M1 悬崖点')
ax2.axhline(y=5, color=C_GRAY, linestyle=':', alpha=0.7, label='5% 基准线')
ax2.set_title('平均留存曲线', fontsize=13, fontweight='bold')
ax2.set_xlabel('距首月月数', fontsize=10)
ax2.set_ylabel('留存率（%）', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_xticks(offsets)

# 图3：复购用户 vs 一次性用户
ax3 = axes[1, 0]
colors_rp = [C_ORANGE, C_MED_BLUE, C_GREEN]
bars3 = ax3.bar(repurchase_df['customer_type'], repurchase_df['customer_count'],
                color=colors_rp, alpha=0.85)
for bar, cnt, pct in zip(bars3, repurchase_df['customer_count'], repurchase_df['customer_pct']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{cnt:,}\n({pct:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax3.set_title('用户复购类型分布', fontsize=13, fontweight='bold')
ax3.set_ylabel('用户数量', fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# 图4：GMV贡献对比
ax4 = axes[1, 1]
x_rp = range(len(repurchase_df))
bars4 = ax4.bar(x_rp, repurchase_df['total_gmv_million'], color=colors_rp, alpha=0.85)
for bar, gmv in zip(bars4, repurchase_df['total_gmv_million']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{gmv:.2f}M', ha='center', fontsize=10, fontweight='bold')
ax4.set_xticks(x_rp)
ax4.set_xticklabels(repurchase_df['customer_type'])
ax4.set_title('各类型用户 GMV 贡献（百万 BRL）', fontsize=13, fontweight='bold')
ax4.set_ylabel('GMV（百万 BRL）', fontsize=10)
ax4.grid(axis='y', alpha=0.3)

plt.suptitle('Olist 留存群组分析', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/Users/cristinayang/Desktop/巴西olist分析项目/08_cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n用户复购类型：")
print(repurchase_df.to_string(index=False))

print("\n前6个群组的M1留存率：")
if cohort_pivot.shape[1] > 1:
    m1_retention = cohort_pivot.iloc[:6, 1]
    for idx, val in m1_retention.items():
        print(f"  {str(idx)[:7]}: {val:.2f}%")

print("\n✅ 图表已保存：08_cohort_retention.png")



In [ ]:
---
## 第九部分：RFM 用户分层

**分析问题**：如何识别高价值客户？各分层的规模和 GMV 贡献如何？

**分析方法**：对 R、F、M 三个指标分别计算后，先转换为 1-5 分得分，再基于 R 和 F 得分交叉分层，M 仅作描述性统计。

**指标定义**：
| 指标 | 计算方式 | 说明 |
|---|---|---|
| **R（Recency）** | DATEDIFF('day', 最后下单日期, '2018-08-26') | 天数越小越活跃 |
| **F（Frequency）** | COUNT(订单ID)，按 customer_unique_id 聚合 | 复购率偏低，F 是忠诚度核心指标 |
| **M（Monetary）** | SUM(支付金额) | 累计 GMV 贡献，仅做描述性统计 |

**R/F 得分规则（1-5 分制）**：

| 指标 | 1分 | 2分 | 3分 | 4分 | 5分 |
|---|---|---|---|---|---|
| R（天数） | >365 | 181-365 | 91-180 | 31-90 | ≤30 |
| F（频次） | 0次 | 1次 | 2次 | 3次 | ≥4次 |

**用户分层逻辑（仅用 R 和 F，M 仅作描述统计）**：

| 分层 | 判定条件 | 核心特征 |
|---|---|---|
| 🥇 重要价值用户 | R得分 ≥ 3 **且** F得分 ≥ 3 | 近期有购买 + 多次购买，平台忠诚核心与主要利润来源 |
| 🌱 重要发展用户 | R得分 ≥ 3 **且** F得分 < 3 | 近期刚下单的新客户，次数少但极具转化潜力 |
| ⚠️ 重要挽留用户 | R得分 < 3 **且** F得分 ≥ 3 | 曾多次购买的老客户，但已很久未活跃，流失风险高 |
| 🪪 一般保持用户 | R得分 < 3 **且** F得分 < 3 | 频率低且长期未下单，低价值或已流失的边缘用户 |

> **注意**：M 不参与分层判定，仅在各分层描述时作为"客单价贡献"指标使用。



In [ ]:
# ============================================================
# 第九部分：RFM 用户分层
# ============================================================

# --- SQL：RFM 计算与 4 组分层（仅用 R + F，M 仅作描述统计） ---
query_rfm = """
WITH customer_rfm_base AS (
    -- Step 1: 计算每个客户的 R/F/M 原始值
    SELECT
        o.customer_unique_id,
        COUNT(DISTINCT o.order_id)                          AS frequency,
        SUM(p.payment_value)                                AS monetary,
        DATEDIFF('day',
                 MAX(o.order_purchase_timestamp::DATE)::DATE,
                 '2018-08-26'::DATE)                       AS recency_days
    FROM olist_orders o
    JOIN olist_payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled')
    GROUP BY 1
),
rfm_scored AS (
    -- Step 2: R/F 分别打 1-5 分
    SELECT
        customer_unique_id,
        frequency,
        monetary,
        recency_days,

        -- R 得分：天数越少分越高（活跃 = 高分）
        CASE
            WHEN recency_days <= 30  THEN 5
            WHEN recency_days <= 90  THEN 4
            WHEN recency_days <= 180 THEN 3
            WHEN recency_days <= 365 THEN 2
            ELSE 1
        END AS r_score,

        -- F 得分：频次越高分越高
        CASE
            WHEN frequency >= 4 THEN 5
            WHEN frequency = 3  THEN 4
            WHEN frequency = 2  THEN 3
            WHEN frequency = 1  THEN 2
            ELSE 1
        END AS f_score

    FROM customer_rfm_base
),
rfm_segmented AS (
    -- Step 3: 仅用 R得分 + F得分 做四象限分层（M 仅描述）
    SELECT
        customer_unique_id,
        frequency,
        monetary,
        recency_days,
        r_score,
        f_score,

        CASE
            WHEN r_score >= 3 AND f_score >= 3 THEN '重要价值用户'
            WHEN r_score >= 3 AND f_score <  3 THEN '重要发展用户'
            WHEN r_score <  3 AND f_score >= 3 THEN '重要挽留用户'
            ELSE                                      '一般保持用户'
        END AS segment

    FROM rfm_scored
)
SELECT
    segment,
    COUNT(*)                                                     AS customer_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1)           AS customer_pct,
    ROUND(SUM(monetary) / 1e6, 3)                              AS total_gmv_million,
    ROUND(100.0 * SUM(monetary) / SUM(SUM(monetary)) OVER(), 1) AS gmv_pct,
    ROUND(AVG(monetary), 2)                                     AS avg_monetary,
    ROUND(AVG(frequency), 2)                                    AS avg_frequency,
    ROUND(AVG(recency_days), 0)                                 AS avg_recency_days,
    ROUND(AVG(r_score), 1)                                      AS avg_r_score,
    ROUND(AVG(f_score), 1)                                      AS avg_f_score
FROM rfm_segmented
GROUP BY segment
ORDER BY gmv_pct DESC
"""

rfm_df = con.execute(query_rfm).df()

# --- 可视化 ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

segment_order = ['重要价值用户', '重要发展用户', '重要挽留用户', '一般保持用户']
seg_colors = {
    '重要价值用户': '#F5A623',
    '重要发展用户': '#2E86AB',
    '重要挽留用户': '#E07B3C',
    '一般保持用户': '#6C757D'
}
palette = [seg_colors[s] for s in segment_order]
rfm_idx = rfm_df.set_index('segment')

# 图1：用户数量分布
ax1 = axes[0, 0]
bars1 = ax1.bar(segment_order,
                [rfm_idx.loc[s, 'customer_count'] for s in segment_order],
                color=palette, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, seg in zip(bars1, segment_order):
    row = rfm_idx.loc[seg]
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{int(row.customer_count):,}\n({row.customer_pct:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('RFM 用户分层 - 用户数量', fontsize=13, fontweight='bold')
ax1.set_ylabel('用户数量', fontsize=10)
ax1.tick_params(axis='x', rotation=15)
ax1.grid(axis='y', alpha=0.3)

# 图2：GMV 贡献
ax2 = axes[0, 1]
bars2 = ax2.bar(segment_order,
                [rfm_idx.loc[s, 'total_gmv_million'] for s in segment_order],
                color=palette, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, seg in zip(bars2, segment_order):
    row = rfm_idx.loc[seg]
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{row.total_gmv_million}M\n({row.gmv_pct:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_title('RFM 用户分层 - GMV 贡献（百万 BRL）', fontsize=13, fontweight='bold')
ax2.set_ylabel('GMV（百万 BRL）', fontsize=10)
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', alpha=0.3)

# 图3：气泡图 - 频次 vs 客单价
ax3 = axes[1, 0]
for seg in segment_order:
    row = rfm_idx.loc[seg]
    size = row.customer_count / 400
    ax3.scatter(row.avg_frequency, row.avg_monetary,
                s=size, c=seg_colors[seg], alpha=0.75,
                edgecolors='white', linewidth=2, zorder=3)
    ax3.annotate(seg, (row.avg_frequency, row.avg_monetary),
                 xytext=(5, 5), textcoords='offset points',
                 fontsize=9, fontweight='bold')
ax3.set_title('分层特征：频次 vs 累计客单价（气泡=人数）', fontsize=13, fontweight='bold')
ax3.set_xlabel('平均购买频次（次）', fontsize=10)
ax3.set_ylabel('平均累计消费金额（BRL）', fontsize=10)
ax3.grid(alpha=0.3)

# 图4：指标汇总表
ax4 = axes[1, 1]
ax4.axis('off')
table_data = []
for seg in segment_order:
    row = rfm_idx.loc[seg]
    table_data.append([
        seg,
        f'{int(row.customer_count):,}',
        f'{row.customer_pct:.1f}%',
        f'{row.gmv_pct:.1f}%',
        f'{row.avg_monetary:.1f}',
        f'{row.avg_frequency:.1f}',
        f'{int(row.avg_recency_days)}天'
    ])
col_labels = ['分层', '用户数', '用户占比', 'GMV占比', '累计AOV(BRL)', '平均频次', '平均R(天)']
table = ax4.table(cellText=table_data, colLabels=col_labels,
                   loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.2)
for j in range(len(col_labels)):
    table[(0, j)].set_facecolor('#1F4E79')
    table[(0, j)].set_text_props(color='white', fontweight='bold')
table[(1, 0)].set_facecolor('#FFF8E7')
ax4.set_title('RFM 分层核心指标汇总', fontsize=13, fontweight='bold', pad=20)

plt.suptitle('RFM 用户分层分析（R×F 四象限模型，4 组）', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 打印明细
print("\n" + "=" * 70)
print("RFM 四层用户分析结果")
print("=" * 70)
print(rfm_df.to_string(index=False))
print()
print("分层说明：")
print("  重要价值用户  = R得分>=3 且 F得分>=3  -> 忠诚核心，高频高活")
print("  重要发展用户  = R得分>=3 且 F得分<3   -> 新客户，极具潜力")
print("  重要挽留用户  = R得分<3  且 F得分>=3  -> 老客户，流失风险")
print("  一般保持用户  = R得分<3  且 F得分<3   -> 低价值或已流失")


---
## 第十部分：策略建议与结论

### 平台核心发现汇总

#### 规模与质量
- **规模**：~10 万笔订单，~9.8 万独立客户，总 GMV 约 1600 万 BRL，AOV ~161 BRL
- **质量**：92% 准时送达率，4.1/5.0 平均评分，91% 好评率

#### 核心瓶颈
1. **复购率极低**：新客首月复购率仅 3-5%，是行业均值的 1/3，是最关键的 GMV 增长杠杆
2. **物流区域性失衡**：SP 以外地区运费高、时效长，RR/AM 等偏远省份履约率不足 89%
3. **高价值客户集中度过高**：18% 的 Champions 用户贡献 50%+ GMV，单一依赖风险大

#### 最大机会点
- 若首月复购率从 5% 提升至 10%，预估每年可新增 GMV **150-250 万 BRL**
- 长分期用户（13-24期）LTV 是全站均值 3.3 倍，是高客单价品类的天然优势群体
- 部分长尾品类（Food&Drinks, Fashion）在物流弱势地区表现出"物流不敏感"特征，适合偏远地区拓展

---

### 策略建议（按优先级排序）

#### P1：【立即启动】首月复购激活
- **目标**：首月复购率 5% → 10-15%
- **手段**：订单确认页发放定向复购券；购物车挽回邮件（24h内触发）；首单后7天推送个性化推荐
- **预期**：每 1% 复购率提升 ≈ 新增 GMV 30-50 万 BRL/年

#### P2：【4周内】配送时效优化
- **目标**：全站平均配送 12.5 天 → 10 天以内；准时率 92% → 96%+
- **重点区域**：AL、AP、RR、AM、PA（5个低效州）
- **手段**：建立商家三段式 SLA 考核（4天/5-7天/>7天分级）；对 Baby&Kids、HomeAppliances 品类实行更严格的出库时限

#### P3：【持续进行】品类质量改善
- **目标**：所有品类平均评分 ≥ 3.8 分
- **重点品类**：review_score < 3.5 的低评分品类（根因分析 + 商家帮扶）
- **手段**：差评率 > 30% 的商家触发自动预警；高敏感品类（母婴、家电）实行"严准入、重赔付"

#### P4：【并行推进】高价值用户保护
- **目标**：Champions/Loyal 客户留存率 ≥ 95%
- **手段**：M 值前 20% 流失用户主动触达 + 时效补偿券；分层权益体系（积分、专属客服、提前购）
- **警示**：流失 VIP 的逾期率是留存组的 1.6 倍，物流问题是核心流失驱动因素

#### P5：【中长期】区域履约差异化策略
- **核心区域（SP）**：选出优质物流合作方，配送时效控制在 5 天以内，优化轻小件运费结构
- **偏远区域**：前端明确展示预计配送时效（20-25天）；建立超时补偿机制，降低用户预期落差
- **运费策略**：以 SP 省为标杆，对 MG 等中等省份实施运费补贴试点（建议先以 $2,000 做 A/B 测试）

---
*分析师：Cristina Yan | 数据周期：2017-1 ~ 2018-08*

